In [1]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from func.UsefulFunctions import prettifyGenres

This notebook takes the group data (that was extracted from the Music Brainz JSON) and inspects it. Groups are groups, choirs, orchestras unless stated otherwise.

In [2]:
groupDF = pl.read_parquet("/home/aja25/Documents/fromCluster/ensemblesMusicBrainzData.parquet")

#drop the relations table as it is no longer needed
groupDF = groupDF.drop("relations")
#De-duplicate the data
groupDF = groupDF.unique()

num_of_ids = groupDF.select(
    pl.col("id")
).unique().count().item(0,0)
print(f"There are {num_of_ids} unique IDs in the dataset (that is a snapshot from 18/07/2026)")


There are 672717 unique IDs in the dataset (that is a snapshot from 18/07/2026)


In [3]:
#How many people do not have a name in this dataset?
num_people_with_no_names = groupDF.select(
    pl.col("name"),
    pl.col("id")
).unique().null_count()
print(num_people_with_no_names)
print(f"There are {num_people_with_no_names.item(0,0)} entities with no name, and {num_people_with_no_names.item(0,1)} entities with no ids.")

shape: (1, 2)
┌──────┬─────┐
│ name ┆ id  │
│ ---  ┆ --- │
│ u32  ┆ u32 │
╞══════╪═════╡
│ 0    ┆ 0   │
└──────┴─────┘
There are 0 entities with no name, and 0 entities with no ids.


In [4]:
#check if there are any duplicates in ids
ids = groupDF.select(
    pl.col("id")
).is_duplicated()
print(ids.value_counts())

print("Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication ")

shape: (1, 2)
┌───────┬────────┐
│       ┆ count  │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 672717 │
└───────┴────────┘
Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication 


In [5]:
#Do a count for the countries for the dataset.
countryCount = groupDF.select(pl.col("country").value_counts(sort=True)).unnest("country")
print(countryCount)

shape: (236, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ null    ┆ 203557 │
│ US      ┆ 121124 │
│ GB      ┆ 45511  │
│ DE      ┆ 33522  │
│ JP      ┆ 29454  │
│ …       ┆ …      │
│ BQ      ┆ 1      │
│ MS      ┆ 1      │
│ SS      ┆ 1      │
│ WF      ┆ 1      │
│ GQ      ┆ 1      │
└─────────┴────────┘


Based off the table, the most common countries are (outside of null which shows that most groups do not have a country attached to them is) US, GB, DE, and japan

In [6]:
#What is the most common type of group for the entities
typeCount = groupDF.group_by("type").agg(pl.col("id").count())
typeCountPercent = groupDF.select(pl.col("type").value_counts(sort=True, normalize=True, name="Fraction")).unnest("type")
print(typeCount)

typeChart = (
    alt.Chart(typeCount).mark_bar().encode(
        y="id",
        x="type",
    )
    .properties(width=500, title="Frequency of Types on Music Brainz")
)
typeChart.encoding.x.title = "Type"
typeChart.encoding.y.title = "Count"
typeChart.show()

shape: (3, 2)
┌───────────┬────────┐
│ type      ┆ id     │
│ ---       ┆ ---    │
│ str       ┆ u32    │
╞═══════════╪════════╡
│ Group     ┆ 650186 │
│ Orchestra ┆ 11760  │
│ Choir     ┆ 10771  │
└───────────┴────────┘


alt.Chart(...)

In [7]:
print(typeCountPercent)

shape: (3, 2)
┌───────────┬──────────┐
│ type      ┆ Fraction │
│ ---       ┆ ---      │
│ str       ┆ f64      │
╞═══════════╪══════════╡
│ Group     ┆ 0.966507 │
│ Orchestra ┆ 0.017481 │
│ Choir     ┆ 0.016011 │
└───────────┴──────────┘


The biggest type in this group is "Group" by a large percent (96.7%), and the other ensemble types exist around 1.7% and 1.6%.

In [8]:
#Let's get the genres out

groupDFPandas = groupDF.to_pandas(use_pyarrow_extension_array=True)
groupDFPandas['genreList'] = groupDFPandas['genres'].map(prettifyGenres)
groupDF = pl.from_pandas(groupDFPandas)
display(groupDF.head(3))

id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
str,str,str,str,str,list[struct[4]],list[str],list[str],struct[3],list[str]
"""d1ce3eb2-cbbe-4292-8ae2-70c2d0…","""Repent""","""Group""",null,"""DE""",[],[],[],"{null,""1992"",false}",[]
"""95409130-bc14-4982-836c-23b3ce…","""Hiatus""","""Group""",null,"""GB""",[],[],[],"{null,""2016-04-30"",false}",[]
"""06c32b77-5cca-486e-9edf-309409…","""Brian Johnson Band""","""Group""",null,"""US""",[],[],[],"{null,null,false}",[]


In [9]:
groupDF.write_parquet("./Data/Network_groupMBData.parquet")

In [10]:
#Get the most common genre
genreCount = groupDF.select(
    pl.col("id"),
    pl.col("name"),
    pl.col("type"),
    pl.col("genreList").alias("Genre"),
)

genreCount = genreCount.explode("Genre", empty_as_null=True, keep_nulls=True)
genreCount = genreCount.unique()
number_of_genres = genreCount.select(
    pl.col("Genre").value_counts(sort=True)
).unnest("Genre")
print(number_of_genres)


shape: (1_339, 2)
┌──────────────────────────────┬────────┐
│ Genre                        ┆ count  │
│ ---                          ┆ ---    │
│ str                          ┆ u32    │
╞══════════════════════════════╪════════╡
│ null                         ┆ 571747 │
│ rock                         ┆ 11429  │
│ punk                         ┆ 9985   │
│ black metal                  ┆ 6911   │
│ metal                        ┆ 6832   │
│ …                            ┆ …      │
│ extratone                    ┆ 1      │
│ psichedelia occulta italiana ┆ 1      │
│ township bubblegum           ┆ 1      │
│ son huasteco                 ┆ 1      │
│ descarga                     ┆ 1      │
└──────────────────────────────┴────────┘


In [11]:
#Look for the type and genre split using group by.
genreCount.group_by(["type",'Genre']).agg(pl.col('id').count())

type,Genre,id
str,str,u32
"""Group""","""chopper""",1
"""Group""","""izvorna bosanska muzika""",2
"""Choir""","""black metal""",1
"""Group""","""punk poetry""",2
"""Group""","""skiffle""",14
…,…,…
"""Group""","""taarab""",1
"""Group""","""uptempo hardcore""",6
"""Group""","""cante alentejano""",1


In [12]:
#Get the group and country count
groupDF.group_by(["type","country"]).agg(pl.col('id').count())

type,country,id
str,str,u32
"""Choir""","""HK""",32
"""Orchestra""","""CZ""",134
"""Group""","""UZ""",43
"""Group""","""CK""",10
"""Group""","""NG""",205
…,…,…
"""Group""","""PF""",21
"""Orchestra""","""LT""",13
"""Orchestra""","""LU""",8
